In [3]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-openai \
    chromadb \
    pypdf

In [2]:
from google.colab import files

uploaded = files.upload()

Saving synthetic_medical_report.pdf to synthetic_medical_report.pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("synthetic_medical_report.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

/tmp/ipykernel_20572/2812516887.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 1


In [4]:
print(documents[0].page_content[:2000])

SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Date
20 August 2026
Chief Complaint
The patient reports persistent fatigue, occasional dizziness, and reduced exercise tolerance for approximately
three weeks.
Vital Signs
Blood Pressure: 138/88 mmHg
Heart Rate: 82 beats/min
Temperature: 98.4 °F
Respiratory Rate: 16 breaths/min
Laboratory Results
Hemoglobin: 11.2 g/dL
White Blood Cell Count: 6,800 /µL
Fasting Glucose: 108 mg/dL
Total Cholesterol: 215 mg/dL
LDL Cholesterol: 142 mg/dL
HDL Cholesterol: 48 mg/dL
Assessment
The findings are consistent with mild anemia. The lipid profile shows borderline elevated total cholesterol and
LDL cholesterol. Fasting glucose is slightly elevated.
Medications
Iron supplement, 1 tablet daily, as prescribed. No other regular medications were reported.
Recommendations
Continue the prescribed iron supplement an

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [6]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk.page_content)


--- CHUNK 1 ---
SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Date
20 August 2026
Chief Complaint
The patient reports persistent fatigue, occasional dizziness, and reduced exercise tolerance for approximately
three weeks.
Vital Signs
Blood Pressure: 138/88 mmHg
Heart Rate: 82 beats/min
Temperature: 98.4 °F
Respiratory Rate: 16 breaths/min
Laboratory Results

--- CHUNK 2 ---
Laboratory Results
Hemoglobin: 11.2 g/dL
White Blood Cell Count: 6,800 /µL
Fasting Glucose: 108 mg/dL
Total Cholesterol: 215 mg/dL
LDL Cholesterol: 142 mg/dL
HDL Cholesterol: 48 mg/dL
Assessment
The findings are consistent with mild anemia. The lipid profile shows borderline elevated total cholesterol and
LDL cholesterol. Fasting glucose is slightly elevated.
Medications
Iron supplement, 1 tablet daily, as prescribed. No other regular medications were reported.
Recom

In [7]:
!pip install -q sentence-transformers langchain-huggingface

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
chunk_text = chunks[0].page_content

vector = embeddings.embed_query(chunk_text)

print("Chunk:")
print(chunk_text)

print("\nVector length:", len(vector))

Chunk:
SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Date
20 August 2026
Chief Complaint
The patient reports persistent fatigue, occasional dizziness, and reduced exercise tolerance for approximately
three weeks.
Vital Signs
Blood Pressure: 138/88 mmHg
Heart Rate: 82 beats/min
Temperature: 98.4 °F
Respiratory Rate: 16 breaths/min
Laboratory Results

Vector length: 384


In [9]:
!pip install -q chromadb langchain-chroma

In [10]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="medical_reports"
)

In [11]:
query = "What is the patient's blood pressure?"

In [12]:
results = vectorstore.similarity_search(
    query,
    k=3
)

In [21]:
for i, result in enumerate(results):
    print(f"\n--- RESULT {i+1} ---")
    print(result.page_content)


--- RESULT 1 ---
SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Date
20 August 2026
Chief Complaint
The patient reports persistent fatigue, occasional dizziness, and reduced exercise tolerance for approximately
three weeks.
Vital Signs
Blood Pressure: 138/88 mmHg
Heart Rate: 82 beats/min
Temperature: 98.4 °F
Respiratory Rate: 16 breaths/min
Laboratory Results

--- RESULT 2 ---
Laboratory Results
Hemoglobin: 11.2 g/dL
White Blood Cell Count: 6,800 /µL
Fasting Glucose: 108 mg/dL
Total Cholesterol: 215 mg/dL
LDL Cholesterol: 142 mg/dL
HDL Cholesterol: 48 mg/dL
Assessment
The findings are consistent with mild anemia. The lipid profile shows borderline elevated total cholesterol and
LDL cholesterol. Fasting glucose is slightly elevated.
Medications
Iron supplement, 1 tablet daily, as prescribed. No other regular medications were reported.
Rec

In [22]:
for i, result in enumerate(results):
    print(f"\n--- RESULT {i+1} ---")
    print("Text:", result.page_content[:200])
    print("Metadata:", result.metadata)


--- RESULT 1 ---
Text: SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Dat
Metadata: {'creator': '(unspecified)', 'moddate': '2026-08-31T18:20:10+00:00', 'author': '(anonymous)', 'producer': 'ReportLab PDF Library - (opensource)', 'keywords': '', 'source': 'synthetic_medical_report.pdf', 'page': 0, 'trapped': '/False', 'subject': '(unspecified)', 'creationdate': '2026-08-31T18:20:10+00:00', 'page_label': '1', 'total_pages': 1, 'title': '(anonymous)'}

--- RESULT 2 ---
Text: Laboratory Results
Hemoglobin: 11.2 g/dL
White Blood Cell Count: 6,800 /µL
Fasting Glucose: 108 mg/dL
Total Cholesterol: 215 mg/dL
LDL Cholesterol: 142 mg/dL
HDL Cholesterol: 48 mg/dL
Assessment
The f
Metadata: {'author': '(anonymous)', 'creator': '(unspecified)', 'moddate': '2026-08-31T18:20:10+00:00', 'creationdate': '2026-08-31T18:20:10+00:00', 'source': 'synthetic_

In [23]:
for i, result in enumerate(results):

    print(f"\n--- RESULT {i+1} ---")

    print("Content:")
    print(result.page_content)

    print("\nSource:")
    print(result.metadata.get("source"))

    print("Page:")
    print(result.metadata.get("page_label"))


--- RESULT 1 ---
Content:
SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Date
20 August 2026
Chief Complaint
The patient reports persistent fatigue, occasional dizziness, and reduced exercise tolerance for approximately
three weeks.
Vital Signs
Blood Pressure: 138/88 mmHg
Heart Rate: 82 beats/min
Temperature: 98.4 °F
Respiratory Rate: 16 breaths/min
Laboratory Results

Source:
synthetic_medical_report.pdf
Page:
1

--- RESULT 2 ---
Content:
Laboratory Results
Hemoglobin: 11.2 g/dL
White Blood Cell Count: 6,800 /µL
Fasting Glucose: 108 mg/dL
Total Cholesterol: 215 mg/dL
LDL Cholesterol: 142 mg/dL
HDL Cholesterol: 48 mg/dL
Assessment
The findings are consistent with mild anemia. The lipid profile shows borderline elevated total cholesterol and
LDL cholesterol. Fasting glucose is slightly elevated.
Medications
Iron supplement, 1 tablet daily

In [13]:
!pip install -q -U langchain-google-genai

In [14]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google Gemini API key: ")

Enter your Google Gemini API key: ··········


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [16]:
response = llm.invoke(
    "What is Retrieval-Augmented Generation in one sentence?"
)

print(response.content)

[{'type': 'text', 'text': '**Retrieval-Augmented Generation (RAG)** is an AI technique that improves the accuracy and reliability of large language models by fetching relevant facts from external knowledge sources before generating a response.', 'extras': {'signature': 'ErYRCrMRARFNMg/hJTUJcqbTseUrFeheFha7GTE1GTpZGiT79M6ZjB2WMVaE/LbkxznKmONXi9Xue3HbdNZWyyTWrGd8sn1jMjtv8Qqf0SlTBajYGKgJBtMRoP/cgWKsl2y7bJY6k4QjrB52XhgMNYbsOCXLimhDz1zwzCBPEZmwi8LuMwPVbRKbiLeeBEO3GCQzz3UFOqLIA1VzTXsQQbLrP/PrOhPt+Zp9LV070N6yTmTYxmkGYsqWlXVwKGVi3haHydwZR7PY5Hk7Rz4TLQE0chJG/DiLkOBiHkbRBwtFMZaVQsjgCuWNZkbJTJ16hz2grBlHeD3FKfdh4I4EMVqaqd6XBo4fuLWFDTSG39M46J5kYRlARntlD8C94gtsa8mN2uqr3QTDtLcPepdUpRWmfNdou1zTDnSOAwN095K0EHtfSWzECBg8c4VKvzgY7Y+0VP+RQwdxMAIWn5MQgcyG5OJfO3Tg+L79+p2SHPDNCvxPKbkbA70oss/RpXlpZNWoiaNzTWG4u3Iid4UZiEbTFMQwtX5lDF/m24dzHbU4yJwrs+sRdP2rpEDZY4FXAKcsQ3KtL1b+9vVb1LkkFVnZlkuqgj+d9OE+xwU7lMSTYmDzjWK1u9MM7aQLjeuea9NOg0uyGT1KqU/sU8fLOj4Ba1vovDtvJS4/Y42E8NmzzDh6I9e7ri3J0h9SevX8XuRbXk8Afpi8kaBGdtYuiiE9Q

In [30]:
context = "\n\n".join(
    result.page_content
    for result in results
)

print(context)

SYNTHETIC MEDICAL REPORT
 For RAG system testing only — fictional patient, not a real medical record
Patient Information
 Patient ID
SYN-2026-001
Name
Alex Morgan
Age
45 years
Gender
Female
Report Date
20 August 2026
Chief Complaint
The patient reports persistent fatigue, occasional dizziness, and reduced exercise tolerance for approximately
three weeks.
Vital Signs
Blood Pressure: 138/88 mmHg
Heart Rate: 82 beats/min
Temperature: 98.4 °F
Respiratory Rate: 16 breaths/min
Laboratory Results

Laboratory Results
Hemoglobin: 11.2 g/dL
White Blood Cell Count: 6,800 /µL
Fasting Glucose: 108 mg/dL
Total Cholesterol: 215 mg/dL
LDL Cholesterol: 142 mg/dL
HDL Cholesterol: 48 mg/dL
Assessment
The findings are consistent with mild anemia. The lipid profile shows borderline elevated total cholesterol and
LDL cholesterol. Fasting glucose is slightly elevated.
Medications
Iron supplement, 1 tablet daily, as prescribed. No other regular medications were reported.
Recommendations

Recommendations
Conti

In [31]:
prompt = f"""
You are a Medical Report Assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context,
say: "The information is not available in the report."

Do not make up medical information.

Context:
{context}

User Question:
{query}
"""

In [35]:
response = llm.invoke(prompt)


answer = response.content[0]["text"]

print(answer)

Based on the provided report, the patient's blood pressure is 138/88 mmHg.


In [36]:
print("Answer:")
answer = response.content[0]["text"]

print(answer)

print("\nSources:")
sources = set()

for result in results:
    source = result.metadata.get("source")
    page = result.metadata.get("page_label")

    sources.add((source, page))

for source, page in sources:
    print(f"- {source} (Page {page})")

Answer:
Based on the provided report, the patient's blood pressure is 138/88 mmHg.

Sources:
- synthetic_medical_report.pdf (Page 1)


In [17]:
def ask_medical_report(query):

    # Step 1: Retrieve relevant chunks
    results = vectorstore.similarity_search(
        query,
        k=3
    )

    # Step 2: Create context
    context = "\n\n".join(
        result.page_content
        for result in results
    )

    # Step 3: Create prompt
    prompt = f"""
You are a Medical Report Assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context,
say: "The information is not available in the report."

Do not make up medical information.

Context:
{context}

User Question:
{query}
"""

    # Step 4: Generate answer
    response = llm.invoke(prompt)

    # Step 5: Extract answer text
    answer = response.content[0]["text"]

    # Step 6: Extract unique sources
    sources = set()

    for result in results:
        source = result.metadata.get("source")
        page = result.metadata.get("page_label")
        sources.add((source, page))

    return answer, sources

In [18]:
answer, sources = ask_medical_report(
    "What is the patient's blood type?"
)

print("Answer:")
print(answer)

print("\nSources:")
for source, page in sources:
    print(f"- {source} (Page {page})")

Answer:
The information is not available in the report.

Sources:
- synthetic_medical_report.pdf (Page 1)
